In [ ]:
! pip install -q transformers sentencepiece accelerate pypdf sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 66.3 MB/s eta 0:00:00


In [ ]:
from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

import faiss
import numpy as np

from transformers import pipeline

In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
file_content = PdfReader("visa_cheklist_2.pdf")
pdf_text = ""
for page in file_content.pages[:1]:
    pdf_text += page.extract_text()


In [ ]:
example_para ="""Machine Learning (ML) is a branch of Artificial Intelligence (AI) that enables computers to learn from data without being explicitly programmed. Instead of following fixed rules, ML algorithms identify patterns in data and make predictions or decisions.

There are three main types of Machine Learning. Supervised Learning uses labeled data to train models. Unsupervised Learning discovers hidden patterns in unlabeled data. Reinforcement Learning trains an agent by rewarding good actions and penalizing bad ones.

Machine Learning has many real-world applications. It is used in healthcare to detect diseases, in finance to identify fraudulent transactions, in e-commerce to recommend products, and in self-driving cars to recognize roads, traffic signs, and pedestrians.

Although Machine Learning is powerful, it also has challenges. Models require high-quality data, sufficient computing power, and regular updates. Poor data quality can lead to inaccurate predictions, making data preprocessing an essential step in every ML project."""

In [ ]:
chunk_list = []
chunk_size = 120

for i in range(0,len(example_para),chunk_size):
    chunk_list.append(example_para[i:i+chunk_size])

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
) #loading the embedding model

chunk_embeddings = embedding_model.encode(chunk_list)

dimension = chunk_embeddings.shape[1] # saving the vectors into FAISS database
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)


In [ ]:
chatbot = pipeline(
    "text-generation",
    model="microsoft/Phi-3-mini-4k-instruct"
)

In [ ]:

question = "What kind of jobs are considered as category A?"

question_embedding = embedding_model.encode([question])

distance, index_number = index.search(
    np.array(question_embedding),
    k=3)

retrieved_chunks = []

for idx in index_number[0]:
    retrieved_chunks.append(chunk_list[idx])

context = "\n\n".join(retrieved_chunks)

prompt = f"""
<|user|>

1. Please use the context and carefully read the question and before answering.
2. Please ignore the characters \n\n
3. The sentence should be meaningful

Use ONLY the context below.

Context:
{context}

Question:
{question}

If the answer is not present, reply exactly:

I couldn't find that information.

<|assistant|>
"""

response = chatbot(
        prompt,
        max_new_tokens=50,
        do_sample=False,
        return_full_text=False
    )
answer = response[0]["generated_text"].strip()
print(answer)